In [1]:
from pyspark.sql.types import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid
 
raw_table = "rentcast_raw_data"   
bronze_delta_path = "Tables/rentcast_bronze_delta"
eto_tracking_path = "Tables/rentcast_bronze_tracking"

StatementMeta(, 47fcd065-837d-4976-bf4f-2d5055dba011, 3, Finished, Available, Finished)

In [2]:
data_schema = StructType([
    StructField("addressLine1", StringType(), True),
    StructField("addressLine2", StringType(), True),
    StructField("bathrooms", DoubleType(), True),
    StructField("bedrooms", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("county", StringType(), True),
    StructField("countyFips", StringType(), True),   
    StructField("createdDate", TimestampType(), True),
    StructField("daysOnMarket", IntegerType(), True),
    StructField("formattedAddress", StringType(), True),
    StructField("history", StringType(), True),
    StructField("hoa", StringType(), True),
    StructField("id", StringType(), True),
    StructField("lastSeenDate", TimestampType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("listedDate", TimestampType(), True),
    StructField("listingAgent", StringType(), True),
    StructField("listingOffice", StringType(), True),
    StructField("listingType", StringType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("lotSize", DoubleType(), True),
    StructField("mlsName", StringType(), True),
    StructField("mlsNumber", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("propertyType", StringType(), True),
    StructField("removedDate", StringType(), True),   
    StructField("squareFootage", IntegerType(), True),
    StructField("state", StringType(), True),
    StructField("stateFips", StringType(), True),   
    StructField("status", StringType(), True),
    StructField("yearBuilt", IntegerType(), True),
    StructField("zipCode", StringType(), True), 
    StructField("extraction_date", TimestampType(), True),
    StructField("offset_used", IntegerType(), True),
    StructField("run_id", LongType(), True)
])

# ETO Tracking Schema
eto_schema = StructType([
    StructField("batch_id", StringType(), False),
    StructField("ingestion_timestamp", TimestampType(), False),
    StructField("source_table", StringType(), False),
    StructField("records_in_batch", LongType(), True),
    StructField("records_ingested", LongType(), True),
    StructField("status", StringType(), False),
    StructField("error_message", StringType(), True),
    StructField("processing_duration_seconds", DoubleType(), True),
    StructField("target_table", StringType(), False),
    StructField("run_ids_processed", StringType(), True),  # Comma-separated run_ids
    StructField("last_extraction_date", TimestampType(), True)
])

contact_schema = StructType([
    StructField("name", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("email", StringType(), True),
    StructField("website", StringType(), True)
])

StatementMeta(, 47fcd065-837d-4976-bf4f-2d5055dba011, 4, Finished, Available, Finished)

In [3]:
def get_last_processed_extraction_date(): 
    if DeltaTable.isDeltaTable(spark, eto_tracking_path):
        result = (
            spark.read.format("delta")
            .load(eto_tracking_path)
            .filter(F.col("status") == "SUCCESS")
            .agg(F.max("last_extraction_date").alias("max_extraction_date"))
            .collect()
        )
        if result and result[0]["max_extraction_date"]:
            return result[0]["max_extraction_date"]
    return None

def parse_json_columns(df): 
    df = df.withColumn("listingAgent_struct", F.from_json(F.col("listingAgent"), contact_schema))
    df = df.withColumn("listingOffice_struct", F.from_json(F.col("listingOffice"), contact_schema))
    
    df = (df
        .withColumn("listingAgent_name", F.col("listingAgent_struct.name"))
        .withColumn("listingAgent_phone", F.col("listingAgent_struct.phone"))
        .withColumn("listingAgent_email", F.col("listingAgent_struct.email"))
        .withColumn("listingAgent_website", F.col("listingAgent_struct.website"))
        .withColumn("listingOffice_name", F.col("listingOffice_struct.name"))
        .withColumn("listingOffice_phone", F.col("listingOffice_struct.phone"))
        .withColumn("listingOffice_email", F.col("listingOffice_struct.email"))
        .withColumn("listingOffice_website", F.col("listingOffice_struct.website"))
        .drop("listingAgent_struct", "listingOffice_struct")
    )
    
    df = df.fillna({
        "listingAgent_name": "", "listingAgent_phone": "", "listingAgent_email": "", "listingAgent_website": "",
        "listingOffice_name": "", "listingOffice_phone": "", "listingOffice_email": "", "listingOffice_website": ""
    })
    
    return df

def log_eto_entry(batch_id, records_in_batch, records_ingested, status, run_ids, last_extraction_date, error_message=None, duration=None):
     
    
    eto_entry = spark.createDataFrame([{
        "batch_id": batch_id,
        "ingestion_timestamp": datetime.now(),
        "source_table": raw_table,
        "records_in_batch": records_in_batch,
        "records_ingested": records_ingested,
        "status": status,
        "error_message": error_message,
        "processing_duration_seconds": duration,
        "target_table": bronze_delta_path,
        "run_ids_processed": run_ids,
        "last_extraction_date": last_extraction_date
    }], schema=eto_schema)
    
    if DeltaTable.isDeltaTable(spark, eto_tracking_path):
        eto_entry.write.format("delta").mode("append").save(eto_tracking_path)
    else:
        eto_entry.write.format("delta").mode("overwrite").save(eto_tracking_path)

def ingest_to_bronze():
    
    
    start_time = datetime.now()
    batch_id = str(uuid.uuid4())
    
    try:
        # Check if raw table exists
        if not spark.catalog.tableExists(raw_table):
            print(f"Source table '{raw_table}' does not exist")
            return
        
        # 1. Get last processed extraction date
        last_processed_date = get_last_processed_extraction_date()
        
        if last_processed_date:
            print(f"Last processed extraction_date: {last_processed_date}")
        else:
            print("No previous processing found - processing all records")
        
        # 2. Read new records from raw table
        raw_df = spark.table(raw_table)
        
        if last_processed_date:
            new_df = raw_df.filter(F.col("extraction_date") > last_processed_date)
        else:
            new_df = raw_df
        
        # Add bronze ingestion metadata
        new_df = new_df.withColumn("bronze_ingestion_timestamp", F.current_timestamp())
        new_df = new_df.withColumn("bronze_batch_id", F.lit(batch_id))
        
        records_in_batch = new_df.count()
        
        if records_in_batch == 0:
            print("No new records to process")
            return
        
        print(f"Processing {records_in_batch} new records")
        print(f"Batch ID: {batch_id}")
        
        # 3. Parse JSON columns
        print("Parsing JSON columns...")
        new_df = parse_json_columns(new_df)
        
        # 4. Get run_ids and max extraction_date for tracking
        run_ids_list = new_df.select("run_id").distinct().rdd.flatMap(lambda x: x).collect()
        run_ids_str = ",".join([str(rid) for rid in run_ids_list])
        max_extraction_date = new_df.agg(F.max("extraction_date")).collect()[0][0]
        
        # 5. Write to bronze
        print("Writing to bronze table...")
        if DeltaTable.isDeltaTable(spark, bronze_delta_path):
            new_df.write.format("delta").mode("append").save(bronze_delta_path)
        else:
            new_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(bronze_delta_path)
        
        # 6. Verify records written
        bronze_df = spark.read.format("delta").load(bronze_delta_path)
        records_ingested = bronze_df.filter(F.col("bronze_batch_id") == batch_id).count()
        
        # 7. Calculate duration
        duration = (datetime.now() - start_time).total_seconds()
        
        # 8. Log to ETO
        if records_in_batch == records_ingested:
            status = "SUCCESS"
            print(f"Successfully processed {records_ingested}/{records_in_batch} records in {duration:.2f}s")
        else:
            status = "PARTIAL"
            print(f"Processed {records_ingested}/{records_in_batch} records in {duration:.2f}s (MISMATCH)")
        
        log_eto_entry(batch_id, records_in_batch, records_ingested, status, run_ids_str, max_extraction_date, None, duration)
        
        print(f"\nSummary:")
        print(f"   Batch ID: {batch_id}")
        print(f"   Status: {status}")
        print(f"   Records: {records_ingested}/{records_in_batch}")
        print(f"   Duration: {duration:.2f}s")
        print(f"   Run IDs: {run_ids_str}")
        print(f"   Max extraction_date: {max_extraction_date}")
        
    except Exception as e:
        error_msg = str(e)
        print(f"Fatal error in ingestion process: {error_msg}")
        
        # Log failure
        log_eto_entry(batch_id, 0, 0, "FAILED", "", None, error_msg, None)
        raise

StatementMeta(, 47fcd065-837d-4976-bf4f-2d5055dba011, 5, Finished, Available, Finished)

In [4]:
if __name__ == "__main__":
    ingest_to_bronze()

StatementMeta(, 47fcd065-837d-4976-bf4f-2d5055dba011, 6, Finished, Available, Finished)

ℹ️  No previous processing found - processing all records
📊 Processing 500 new records
🔖 Batch ID: 7b11cf00-6b94-4d65-a766-716496e8e02b
🔄 Parsing JSON columns...
💾 Writing to bronze table...
✅ Successfully processed 500/500 records in 36.89s

📋 Summary:
   Batch ID: 7b11cf00-6b94-4d65-a766-716496e8e02b
   Status: SUCCESS
   Records: 500/500
   Duration: 36.89s
   Run IDs: 20251115010128
   Max extraction_date: 2025-11-15 01:01:31.392980
